# Embedding Comparison Analysis
1. Global Overview (Cosine & Euclidean)
2. Detailed Deep-Dive


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
reports_base = Path('../reports')
data_root = Path('../datalocal')
ref_root_default = data_root / 'PC-GITA_v260210_24kHz'
sns.set_theme(style='whitegrid')


In [ ]:
def show_global():
    rows_cos = []
    rows_euc = []
    if not reports_base.exists(): return
    dirs = sorted([d for d in reports_base.iterdir() if d.is_dir() and d.name.startswith('comparison_')])
    for rd in dirs:
        p = rd / 'speaker_embedding_summary.csv'
        if p.exists():
            sdf = pd.read_csv(p)
            sdf.columns = [c.strip() for c in sdf.columns]
            sdf = sdf.rename(columns={'avg_distance': 'avg_cos_dist'})
            parts = rd.name.replace('comparison_', '').split('_')
            name = '_'.join(parts[-2:]) if len(parts) >= 2 else rd.name
            for metric, row_list, col in [('avg_cos_dist', rows_cos, 'Cos'), ('avg_euc_dist', rows_euc, 'Euc')]:
                if metric not in sdf.columns: continue
                overall = sdf.groupby('task')[metric].mean().to_dict()
                for t, v in overall.items():
                    row_list.append({'experiment': name, 'metric': '1. Overall', 'task': t, 'value': v})
                groups = sdf.groupby(['task', 'status'])[metric].mean().unstack()
                if 'HC' in groups.columns:
                    for t, v in groups['HC'].dropna().to_dict().items():
                        row_list.append({'experiment': name, 'metric': '2. HC', 'task': t, 'value': v})
                if 'PD' in groups.columns:
                    for t, v in groups['PD'].dropna().to_dict().items():
                        row_list.append({'experiment': name, 'metric': '3. PD', 'task': t, 'value': v})
    for title, rows, cmap in [('Cosine Distance', rows_cos, 'YlOrRd'), ('Euclidean Distance', rows_euc, 'Blues')]:
        if rows:
            display(Markdown(f'### {title} Overview'))
            df = pd.DataFrame(rows)
            pvt = df.pivot(index=['experiment', 'metric'], columns='task', values='value')
            display(pvt.style.background_gradient(cmap=cmap).format('{:.4f}'))
show_global()


## 2. Detailed Experiment Deep-Dive

### Understanding the Metrics
We analyze the embedding space by comparing different types of distance pairs:

#### 1. Intra-speaker Comparison (Fidelity)
- **Ref-Test (Fidelity Same Spk)**: This metric compares parallel samples of the **SAME** speaker (natural vs synthetic). It measures how accurately the model reproduces a specific person's voice characteristics. **Goal**: Values should be as close to **0.0** as possible.

#### 2. Inter-speaker Comparison (Baselines)
- **Ref-Ref (Between Natural Spk)**: Average distance between **DIFFERENT** natural speakers for the same sentence. Represents natural variation.
- **Test-Test (Between Synthetic Spk)**: Distance between **DIFFERENT** synthetic voices generated by the model.

**Key Interpretation**: If **Ref-Test** (Fidelity) is significantly lower than **Ref-Ref** (Natural baseline), the synthesis successfully captures individual identity.


In [ ]:
available = sorted([d.name for d in reports_base.glob('comparison_*')]) if reports_base.exists() else []
drop = widgets.Dropdown(options=available, description='Select Exp:')
out = widgets.Output()
def calc_inter(root, smap, lbl):
    rows = []
    files = list(root.glob('*sentence*.pt'))
    for i in range(1, 11):
        sid = 'sentence' + str(i)
        match = [f for f in files if (sid + '_') in f.name or f.name.endswith(sid + '.pt')]
        for st in ['HC', 'PD']:
            gf = [f for f in match if smap.get(f.name.split('_')[0]) == st]
            if len(gf) < 2: continue
            ts = torch.stack([torch.load(f, map_location='cpu').flatten() for f in gf])
            nm = F.normalize(ts, p=2, dim=1)
            idx = torch.triu_indices(len(gf), len(gf), 1)
            cos = 1.0 - torch.mm(nm, nm.t())[idx[0], idx[1]]
            euc = torch.cdist(ts.unsqueeze(0), ts.unsqueeze(0), p=2).squeeze(0)[idx[0], idx[1]]
            for c, e in zip(cos.tolist(), euc.tolist()):
                rows.append({'sentence_id': sid, 'status': st, 'type': lbl, 'cosine_distance': c, 'euclidean_distance': e})
    return pd.DataFrame(rows)
def run(name):
    with out:
        clear_output(wait=True); rd = reports_base / name
        if not (rd / 'speaker_embedding_summary.csv').exists(): return
        sum_df = pd.read_csv(rd / 'speaker_embedding_summary.csv'); sum_df.columns = [c.strip() for c in sum_df.columns]
        smap = {}
        if (rd / 'comparison_integrity.csv').exists():
            idf = pd.read_csv(rd / 'comparison_integrity.csv'); smap = dict(zip(idf['speaker_id'], idf['status']))
        if (rd / 'comparison_embeddings.csv').exists():
            det = pd.read_csv(rd / 'comparison_embeddings.csv')
            sent = det[det['task'] == 'sentences_cleaned'].copy()
            if not sent.empty:
                sent['sentence_id'] = sent['filename'].apply(lambda x: 'sentence' + x.split('sentence')[1].split('_')[0])
                ref_t = sent[['sentence_id', 'status', 'cosine_distance', 'euclidean_distance']].copy()
                ref_t['type'] = 'Ref-Test (Fidelity Same Spk)'
                rr = calc_inter(ref_root_default / 'speaker_embeddings' / 'wavLM', smap, 'Ref-Ref (Between Natural Spk)')
                tn = name.replace('comparison_', '').replace(ref_root_default.name + '_', '')
                tt = calc_inter(data_root / tn / 'speaker_embeddings' / 'wavLM', smap, 'Test-Test (Between Synthetic Spk)')
                comb = pd.concat([ref_t, rr, tt])
                comb['sentence_id'] = pd.Categorical(comb['sentence_id'], [f'sentence{i}' for i in range(1, 11)], True)
                # Separate Cosine Plot
                plt.figure(figsize=(12, 5))
                sns.lineplot(data=comb, x='sentence_id', y='cosine_distance', hue='type', style='status', markers=True)
                plt.title('Cosine Fidelity vs Baselines (95% CI shadows)')
                plt.ylabel('Cosine Distance')
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.show()
                # Separate Euclidean Plot
                plt.figure(figsize=(12, 5))
                sns.lineplot(data=comb, x='sentence_id', y='euclidean_distance', hue='type', style='status', markers=True)
                plt.title('Euclidean Fidelity vs Baselines (95% CI shadows)')
                plt.ylabel('Euclidean Distance')
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.show()
        p_idx = [c for c in ['speaker_id', 'status'] if c in sum_df.columns]
        display(sum_df.pivot(index=p_idx, columns='task', values='avg_cos_dist' if 'avg_cos_dist' in sum_df.columns else 'avg_distance').style.background_gradient(cmap='YlOrRd'))
drop.observe(lambda c: run(c['new']) if c['type']=='change' and c['name']=='value' else None)
display(drop, out); run(drop.options[0])
